<a href="https://colab.research.google.com/github/Alka709/DeepLearning_Projects/blob/main/PDF_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install PyPDF2 sentence-transformers faiss-cpu transformers torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 33.6 MB/s eta 0:00:00


In [2]:
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np


In [3]:
def load_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text

    return text


In [5]:
pdf_text = load_pdf_text("/content/drdo.pdf")
print(pdf_text[:1000])


Advertisement No. DGRE/DRPM/TCHR/5205/Voc Trg/2025-02
La  st Dat  e:    15th Dec 2025
ADVERTISEMENT FOR THE PAID INTERNSHIP FOR ENGINEERING/SCIENCE 
UG & PG STUDENTS
Defence Geoinformatics Research Establishment(DGRE) is  one of  the premier 
Laboratory  of the Defence R&D Organization (DRDO). DGRE invites applications  from 
students (Indian citizens) with excellent academics record for the Paid Internship for a period of 
Six months. Applications are invited from eligible students pursuing under-graduation/post-
graduation in  engineering/science in the prescribed format, latest by 15th Dec 2025, for the 
following disciplines:
Category (a): For Under-Graduate Internship (Students pursuing B.Tech): Internship 
must be a part of their degree courses
Branch
CodeBranch/DisciplineNo.of
VacanciesMonthly
Stipend
(in Rs)Durationof
Internship/ 
Project workLocation of Internship
CSEComputer Science &
Engineering025,000/-06 monthsDGRE, Plot No. 01,
Sector 37-A, 
Chandigarh
CIECivil Engineerin

In [6]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap

    return chunks


In [7]:
chunks = chunk_text(pdf_text)
print("Number of chunks:", len(chunks))
print(chunks[0])


Number of chunks: 28
Advertisement No. DGRE/DRPM/TCHR/5205/Voc Trg/2025-02
La  st Dat  e:    15th Dec 2025
ADVERTISEMENT FOR THE PAID INTERNSHIP FOR ENGINEERING/SCIENCE 
UG & PG STUDENTS
Defence Geoinformatics Research Establishment(DGRE) is  one of  the premier 
Laboratory  of the Defence R&D Organization (DRDO). DGRE invites applications  from 
students (Indian citizens) with excellent academics record for the Paid Internship for a period of 
Six months. Applications are invited from eligible students pursuing und


In [8]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
embeddings = embedding_model.encode(chunks)
print(embeddings.shape)


(28, 384)


In [10]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)


In [11]:
index.add(np.array(embeddings))
print("Vectors stored:", index.ntotal)


Vectors stored: 28


In [12]:
def retrieve_chunks(query, top_k=3):
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), top_k)

    return [chunks[i] for i in indices[0]]


In [13]:
test_chunks = retrieve_chunks("What is machine learning?")
print(test_chunks)


['Advertisement No. DGRE/DRPM/TCHR/5205/Voc Trg/2025-02\nLa  st Dat  e:    15th Dec 2025\nADVERTISEMENT FOR THE PAID INTERNSHIP FOR ENGINEERING/SCIENCE \nUG & PG STUDENTS\nDefence Geoinformatics Research Establishment(DGRE) is  one of  the premier \nLaboratory  of the Defence R&D Organization (DRDO). DGRE invites applications  from \nstudents (Indian citizens) with excellent academics record for the Paid Internship for a period of \nSix months. Applications are invited from eligible students pursuing und', ' Development Organisation\nDefence Geoinformatics Research Establishment\nHim Parisar, Sector 37-A\n Chandigarh-160 036\nTel: 0172-2699804\nFax: 0172-2699802\n1.MINIMUM EDUCATION QUALIFICATIONS: -\nPursuing Graduate/Post Graduate in Engineering and Science, full time course in the \nrespective discipline from a recognized Indian University/Institute.\n2.DURATION OF INTERNSHIP: -\nThe duration of internship/ project work will be for a period of 06 months. The student \nwill  require 

In [14]:
llm = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=512
)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [15]:
def generate_answer(query):
    retrieved_chunks = retrieve_chunks(query)
    context = "\n".join(retrieved_chunks)

    prompt = f"""
    Answer the question using ONLY the context below.
    If the answer is not present, say "I don't know."

    Context:
    {context}

    Question:
    {query}
    """

    response = llm(prompt)
    return response[0]["generated_text"]


In [16]:
while True:
    query = input("Ask a question (type 'exit' to quit): ")

    if query.lower() == "exit":
        break

    answer = generate_answer(query)
    print("\nAnswer:", answer, "\n")


Ask a question (type 'exit' to quit): what is drdo

Answer: Defence Geoinformatics Research Establishment(DGRE) is one of the premier Laboratory of the Defence R&D Organization (DRDO). 

Ask a question (type 'exit' to quit): what is the eligibility criteria for insternship

Answer: CGPA of all previous semesters/ Percentage of marks of all previous semesters/ years and Online/ telephonic interview / interaction as required, subject to satisfactory verification of the documents 

Ask a question (type 'exit' to quit): exit
